In [2]:
import fastf1
import pandas as pd
from pathlib import Path

fastf1.Cache.enable_cache('../cache')

In [ ]:


# --------------------------------------------------
# Setup
# --------------------------------------------------

year = 2025

ROOT_DIR = Path.cwd().parent

print(f"Project Root: {ROOT_DIR}")

raw_folder = (
    ROOT_DIR
    / "data"
    / "raw"
    / str(year)
)

print(f"Raw Folder: {raw_folder}")

if not raw_folder.exists():
    raise FileNotFoundError(
        f"Could not find {raw_folder}"
    )

# --------------------------------------------------
# Find races
# --------------------------------------------------

races = [
    race
    for race in raw_folder.iterdir()
    if race.is_dir()
]

print(f"\nFound {len(races)} races")

if len(races) == 0:
    raise ValueError(
        "No race folders found"
    )

all_stints = []

# --------------------------------------------------
# Process races
# --------------------------------------------------

for race_folder in races:

    race_name = race_folder.name

    print("\n--------------------------------")
    print(f"Processing {race_name}")

    laps_file = race_folder / "laps.csv"

    if not laps_file.exists():

        print(
            f"Skipping {race_name} "
            f"(laps.csv not found)"
        )

        continue

    laps = pd.read_csv(laps_file)

    print(
        f"Loaded {len(laps)} rows"
    )

    required_columns = [
        "Driver",
        "Stint",
        "Compound",
        "LapNumber",
        "LapTime"
    ]

    missing_columns = [
        col
        for col in required_columns
        if col not in laps.columns
    ]

    if missing_columns:

        print(
            f"Missing columns: "
            f"{missing_columns}"
        )

        continue

    # ------------------------------
    # Cleaning
    # ------------------------------

    before_rows = len(laps)

    laps = laps[
        laps["LapTime"].notna()
    ]

    after_rows = len(laps)

    print(
        f"Removed "
        f"{before_rows - after_rows} "
        f"rows with missing LapTime"
    )

    if laps.empty:

        print(
            f"No valid lap data "
            f"for {race_name}"
        )

        continue

    # ------------------------------
    # Transformation
    # ------------------------------

    stints = (
        laps
        .groupby(
            [
                "Driver",
                "Stint",
                "Compound"
            ]
        )
        .agg(
            Laps=(
                "LapNumber",
                "count"
            )
        )
        .reset_index()
    )

    stints["Race"] = race_name

    print(
        f"Created "
        f"{len(stints)} stint records"
    )

    all_stints.append(stints)

# --------------------------------------------------
# Combine all races
# --------------------------------------------------

if len(all_stints) == 0:

    raise ValueError(
        "No stint data generated"
    )

driver_stints = pd.concat(
    all_stints,
    ignore_index=True
)

print("\n--------------------------------")
print("Transformation Complete")

print(
    f"Total Records: "
    f"{len(driver_stints)}"
)

print("\nPreview:")

display(driver_stints.head())

# --------------------------------------------------
# Save
# --------------------------------------------------

output_folder = (
    ROOT_DIR
    / "data"
    / "processed"
)

output_folder.mkdir(
    parents=True,
    exist_ok=True
)

output_file = (
    output_folder
    / "fact_driver_stints.csv"
)

driver_stints.to_csv(
    output_file,
    index=False
)

print(
    f"\nSaved to:"
)
print(output_file)

print(
    f"\nRows Saved:"
)
print(len(driver_stints))


Project Root: d:\Projects\formula1-data-analysis
Raw Folder: d:\Projects\formula1-data-analysis\data\raw\2025

Found 24 races

--------------------------------
Processing abu_dhabi
Loaded 1156 rows
Removed 0 rows with missing LapTime
Created 47 stint records

--------------------------------
Processing australian
Loaded 927 rows
Removed 69 rows with missing LapTime
Created 65 stint records

--------------------------------
Processing austrian
Loaded 1126 rows
Removed 2 rows with missing LapTime
Created 49 stint records

--------------------------------
Processing azerbaijan
Loaded 968 rows
Removed 58 rows with missing LapTime
Created 39 stint records

--------------------------------
Processing bahrain
Loaded 1128 rows
Removed 13 rows with missing LapTime
Created 62 stint records

--------------------------------
Processing belgian
Loaded 879 rows
Removed 60 rows with missing LapTime
Created 42 stint records

--------------------------------
Processing british
Loaded 825 rows
Removed 9

,Driver,Stint,Compound,Laps,Race
0,ALB,1.0,SOFT,8,abu_dhabi
1,ALB,2.0,HARD,25,abu_dhabi
2,ALB,3.0,MEDIUM,25,abu_dhabi
3,ALO,1.0,MEDIUM,16,abu_dhabi
4,ALO,2.0,HARD,42,abu_dhabi



Saved to:
d:\Projects\formula1-data-analysis\data\processed\fact_driver_stints.csv

Rows Saved:
1243
